In [1]:
print(3)

3


In [2]:
pinecode_api_key = "pcsk_38q1oD_UFiAgaqxc9RTv4stsZBuqVnYwG1PwakszjeGHvXzRXw5SqgWFB6QbiyNVur7sab"

In [3]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [6]:
import os
from pinecone import Pinecone, ServerlessSpec
index_name = "hybrid-search-index"
pc = Pinecone(api_key=pinecode_api_key)

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name = index_name,
        dimension = 384,
        metric = "dotproduct",
        spec = ServerlessSpec(cloud = "aws", region = "us-east-1")
    )

In [8]:
index = pc.Index(index_name)
index

In [16]:
#vector embedding and sparse matrix
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")

from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [18]:
from pinecone_text.sparse import BM25Encoder
bm25_encoder = BM25Encoder().default()
bm25_encoder

In [19]:
sentences = [
    "In 2024, I visited Paris",
    "In 2023, I went to London",
    "In 2022, I traveled to New York",
    "In 2021, I explored Tokyo"
]

# tf-idf sparse vectors
bm25_encoder.fit(sentences)

# store the values to a json file
bm25_encoder.dump("bm25_encoder.json")

# load the values from a json file
bm25_encoder = BM25Encoder().load("bm25_encoder.json")

100%|██████████| 4/4 [00:00<00:00, 11.58it/s]


In [20]:
# retrieve sparse vectors
retriever = PineconeHybridSearchRetriever(embeddings=embeddings,index=index, sparse_encoder=bm25_encoder)

In [21]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x0000019683BE17B0>, index=<pinecone.db_data.index.Index object at 0x000001968349FFA0>)

In [22]:
# add documents to the index
retriever.add_texts(
    [
    "In 2024, I visited Paris",
    "In 2023, I went to London",
    "In 2022, I traveled to New York",
    "In 2021, I explored Tokyo"
])

100%|██████████| 1/1 [00:02<00:00,  2.05s/it]


In [23]:
retriever.invoke("What city did i visit in 2024")

[Document(metadata={'score': 0.518513322}, page_content='In 2024, I visited Paris'),
 Document(metadata={'score': 0.355243206}, page_content='In 2023, I went to London'),
 Document(metadata={'score': 0.341648579}, page_content='In 2022, I traveled to New York'),
 Document(metadata={'score': 0.269053936}, page_content='In 2021, I explored Tokyo')]